# Flan-T5 · Fine-tuning on Spider (continued from WikiSQL)

Loads the Flan-T5 checkpoint produced by `Flan-T5-WikiSQL.ipynb` and continues fine-tuning it on the [Spider](https://yale-lily.github.io/spider) cross-domain text-to-SQL dataset. Ends with an interactive Gradio demo.

**Environment:** Built for **Kaggle** (GPU T4 x2 accelerator). Uses `/kaggle/working/` for outputs.

**Requires as Kaggle inputs:**
- The WikiSQL notebook's output, added via *Notebook → Add Input → Your Work*
- A dataset containing Spider's `tables.json` schema file (see the error message in the code cell below for the manual steps if you don't already have one)

To run elsewhere (Colab / local), point `wikisql_local` at a local Flan-T5 checkpoint directory and replace the `/kaggle/input/` discovery logic with a direct path.

## Pipeline
1. Locate and load the WikiSQL checkpoint
2. Load the Spider dataset and its DB schemas
3. Tokenize (question + schema) → SQL pairs
4. Fine-tune with `Seq2SeqTrainer`
5. Evaluate exact-match accuracy on the Spider validation split
6. Launch a Gradio demo for interactive querying


In [ ]:
import subprocess
subprocess.run(
    ["pip", "install", "-q", "-U",
     "transformers", "datasets", "evaluate", "gradio",
     "accelerate", "sentencepiece", "pandas"],
    check=True,
)

import os, re, json, math, random, sqlite3
import glob, shutil, urllib.request
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, Dataset, load_dataset as _lds
try:
    from IPython.display import display
except ImportError:
    display = print
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

os.environ["WANDB_DISABLED"]         = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Output paths — everything saves here (downloadable after session)
LOCAL_MODEL_COPY  = "./wikisql-model-local"   # local copy of WikiSQL model
SPIDER_FINAL_DIR  = "/kaggle/working/spider-flan-t5-final"
SPIDER_OUTPUT_DIR = "/kaggle/working/spider-flan-t5-ckpts"
SPIDER_TABLES_PATH = "/tmp/spider_tables.json"

SPIDER_MAX_INPUT_LENGTH  = 512
SPIDER_MAX_TARGET_LENGTH = 256

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2")

device = "cuda"
print(f"GPUs : {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}")


## 1. Locate and load the WikiSQL checkpoint
Searches `/kaggle/input/` for the WikiSQL model files, copies them to a short local path (avoids Hugging Face repo-id path-length issues), and loads the tokenizer.

In [ ]:

def _find_and_copy_model() -> str:
    # check all possible locations
    candidates = []

    # Case A: flat — model files directly inside /kaggle/input/<slug>/
    for _slug_dir in sorted(glob.glob("/kaggle/input/*/")):
        if (os.path.exists(os.path.join(_slug_dir, "model.safetensors")) and
                os.path.exists(os.path.join(_slug_dir, "config.json"))):
            candidates.append(_slug_dir.rstrip("/"))

    # Case B: one subfolder deep — /kaggle/input/<slug>/<subfolder>/
    for _f in glob.glob("/kaggle/input/**/model.safetensors", recursive=True):
        _d = os.path.dirname(_f)
        if os.path.exists(os.path.join(_d, "config.json")):
            if _d not in candidates:
                candidates.append(_d)

    if not candidates:
        raise RuntimeError(
            "No WikiSQL model found in /kaggle/input/.\n"
            "Make sure you added the WikiSQL notebook output as input:\n"
            "  Notebook -> Add Input -> Your Work -> select the WikiSQL notebook."
        )

    src = candidates[0]
    print(f"Found model at: {src}")
    print(f"Files         : {os.listdir(src)}")

    # copy to local short path
    # Avoids HuggingFace repo-id validation on deep /kaggle/input/ paths
    if os.path.exists(LOCAL_MODEL_COPY):
        print("Local model copy already exists, skipping copy.")
    else:
        print(f"Copying model to {LOCAL_MODEL_COPY} ...")
        shutil.copytree(src, LOCAL_MODEL_COPY)
        print("Copy done.")

    return LOCAL_MODEL_COPY


wikisql_local = _find_and_copy_model()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(wikisql_local)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")


## 2. Load Spider dataset and schemas
Loads Spider from the Hugging Face Hub and builds a `{db_id: {table: [(column, type), ...]}}` schema dictionary from `tables.json`.

In [ ]:
import datasets as _ds_module

print("Loading Spider dataset ...")
spider_raw = _lds("spider")
print(spider_raw)

# find tables.json
_found = None
for _pattern in [
    "/kaggle/input/**/tables.json",
    "/kaggle/input/*/tables.json",
]:
    _hits = glob.glob(_pattern, recursive=True)
    if _hits:
        shutil.copy(_hits[0], SPIDER_TABLES_PATH)
        _found = SPIDER_TABLES_PATH
        print(f"Found tables.json in Kaggle input: {_hits[0]}")
        break

if not _found:
    raise RuntimeError(
        "Could not obtain tables.json.\n\n"
        "MANUAL FIX:\n"
        "  1. Download tables.json from https://github.com/taoyds/spider\n"
        "  2. Upload it as a Kaggle dataset (New Dataset -> upload tables.json)\n"
        "  3. Add that dataset as input to this notebook\n"
        "  4. Re-run — the script will find it under /kaggle/input/"
    )

# build schema dict
with open(SPIDER_TABLES_PATH, encoding="utf-8") as _f:
    _spider_tables = json.load(_f)

spider_schema_dict = {}
for _db in _spider_tables:
    _db_id  = _db["db_id"]
    _tnames = _db.get("table_names_original", _db.get("table_names", []))
    _cols   = _db.get("column_names_original", _db.get("column_names", []))
    _ctypes = _db.get("column_types", [])
    _tables = {t: [] for t in _tnames}
    for (_tidx, _cname), _ctype in zip(_cols, _ctypes):
        if 0 <= _tidx < len(_tnames):
            _tables[_tnames[_tidx]].append((_cname, _ctype))
    spider_schema_dict[_db_id] = _tables



## 3. Tokenize
Builds the `question + db schema → SQL` training pairs and tokenizes them.

In [ ]:
def spider_schema_str(db_id, schema_dict, max_tables=7):
    tables = schema_dict.get(db_id, {})
    if not tables:
        return f"db: {db_id}"
    parts = [
        f"{tn}(" + ", ".join(f"{c} {t}" for c, t in cols[:12]) + ")"
        for tn, cols in list(tables.items())[:max_tables]
    ]
    return "tables: " + " | ".join(parts)


def spider_example_to_input(example, schema_dict):
    return (f"translate to SQL: question: {example['question']} "
            f"| db: {example['db_id']} "
            f"| {spider_schema_str(example['db_id'], schema_dict)}")


def spider_example_to_sql(example):
    return example["query"]


def spider_preprocess_batch(batch):
    inputs, targets = [], []
    for i in range(len(batch["question"])):
        ex = {
            "question": batch["question"][i],
            "db_id":    batch["db_id"][i],
            "query":    batch["query"][i],
        }
        inputs.append(spider_example_to_input(ex, spider_schema_dict))
        targets.append(spider_example_to_sql(ex))
    model_inputs = tokenizer(
        inputs, max_length=SPIDER_MAX_INPUT_LENGTH,
        truncation=True, padding="max_length",
    )
    label_enc = tokenizer(
        targets, max_length=SPIDER_MAX_TARGET_LENGTH,
        truncation=True, padding="max_length",
    )
    model_inputs["labels"] = [
        [t if t != tokenizer.pad_token_id else -100 for t in row]
        for row in label_enc["input_ids"]
    ]
    return model_inputs


spider_tokenized = spider_raw.map(
    spider_preprocess_batch, batched=True,
    remove_columns=spider_raw["train"].column_names,
    desc="Tokenizing Spider",
)
print("Spider tokenized:", spider_tokenized)

_s = spider_raw["train"][0]
print(f"\nSample input : {spider_example_to_input(_s, spider_schema_dict)[:120]} ...")
print(f"Sample target: {spider_example_to_sql(_s)}")

## 4. Fine-tune on Spider
Loads the WikiSQL checkpoint as the starting point and continues training with `Seq2SeqTrainer`.

In [ ]:
# %% 5. LOAD WIKISQL MODEL + TRAIN ON SPIDER
print(f"Loading WikiSQL model from: {wikisql_local}")
spider_model = AutoModelForSeq2SeqLM.from_pretrained(wikisql_local)
spider_model.config.use_cache              = False
spider_model.config.pad_token_id           = tokenizer.pad_token_id
spider_model.config.eos_token_id           = tokenizer.eos_token_id
spider_model.config.decoder_start_token_id = tokenizer.pad_token_id
if hasattr(spider_model, "generation_config"):
    spider_model.generation_config.pad_token_id          = tokenizer.pad_token_id
    spider_model.generation_config.eos_token_id           = tokenizer.eos_token_id
    spider_model.generation_config.decoder_start_token_id = tokenizer.pad_token_id
    spider_model.generation_config.forced_eos_token_id    = tokenizer.eos_token_id
print("WikiSQL model loaded. Single GPU, no DataParallel.")

SPIDER_LR     = 5e-5
SPIDER_EPOCHS = 10
SPIDER_WARMUP = 0.1

data_collator_spider = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=spider_model,
    label_pad_token_id=-100, pad_to_multiple_of=8,
)

_n      = len(spider_tokenized["train"])
_sep    = math.ceil(_n / (2 * 16))   # batch=2, grad_accum=16 → effective batch=32
_total  = _sep * SPIDER_EPOCHS
_warmup = max(1, int(_total * SPIDER_WARMUP))
print(f"steps/epoch={_sep}  total={_total}  warmup={_warmup}")

_skwargs = dict(
    output_dir=SPIDER_OUTPUT_DIR,
    learning_rate=SPIDER_LR,
    per_device_train_batch_size=2,      # small batch — single GPU with 512-token inputs
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,     # keeps effective batch = 32
    gradient_checkpointing=True,        # trades compute for memory
    num_train_epochs=SPIDER_EPOCHS,
    warmup_steps=_warmup,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=SPIDER_MAX_TARGET_LENGTH,
    generation_num_beams=4,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
try:
    _sargs = Seq2SeqTrainingArguments(eval_strategy="epoch", **_skwargs)
except TypeError:
    _sargs = Seq2SeqTrainingArguments(evaluation_strategy="epoch", **_skwargs)

try:
    spider_trainer = Seq2SeqTrainer(
        model=spider_model, args=_sargs,
        train_dataset=spider_tokenized["train"],
        eval_dataset=spider_tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator_spider,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
except TypeError:
    spider_trainer = Seq2SeqTrainer(
        model=spider_model, args=_sargs,
        train_dataset=spider_tokenized["train"],
        eval_dataset=spider_tokenized["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator_spider,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

print(f"\nTraining on {_n:,} Spider examples (single GPU) ...")
print(f"Saving Spider model to: {SPIDER_FINAL_DIR}\n")
spider_trainer.train()
spider_model.config.use_cache = True
spider_trainer.save_model(SPIDER_FINAL_DIR)
tokenizer.save_pretrained(SPIDER_FINAL_DIR)
print(f"\nSpider model saved -> {SPIDER_FINAL_DIR}")
print(f"Files: {os.listdir(SPIDER_FINAL_DIR)}")

## 5. Evaluate on the Spider validation set
Generates predictions with beam search and reports exact-match accuracy; saves predictions to `spider_predictions.csv`.

In [ ]:
spider_model.eval()
spider_preds = []
_val_list    = list(spider_raw["validation"])
_EVAL_BATCH  = 16

print(f"Evaluating on {len(_val_list):,} Spider val examples ...")
for _start in range(0, len(_val_list), _EVAL_BATCH):
    _batch   = _val_list[_start: _start + _EVAL_BATCH]
    _prompts = [spider_example_to_input(e, spider_schema_dict) for e in _batch]
    _enc = tokenizer(
        _prompts, return_tensors="pt",
        max_length=SPIDER_MAX_INPUT_LENGTH,
        truncation=True, padding=True,
    ).to(device)
    with torch.no_grad():
        _out = spider_model.generate(
            **_enc,
            max_length=SPIDER_MAX_TARGET_LENGTH,
            num_beams=4, early_stopping=True,
            decoder_start_token_id=spider_model.config.decoder_start_token_id,
        )
    spider_preds.extend(tokenizer.batch_decode(_out, skip_special_tokens=True))
    if _start % (_EVAL_BATCH * 10) == 0 or _start + _EVAL_BATCH >= len(_val_list):
        print(f"  {min(_start + _EVAL_BATCH, len(_val_list))}/{len(_val_list)} done ...")

_gold = [e["query"] for e in _val_list]
_em   = float(np.mean([
    " ".join(p.lower().split()) == " ".join(g.lower().split())
    for p, g in zip(spider_preds, _gold)
]))
_results = {
    "spider_val_examples": len(_gold),
    "spider_exact_match":  round(_em, 4),
}
print(json.dumps(_results, indent=2))

_preview = pd.DataFrame({
    "db_id":    [e["db_id"]    for e in _val_list[:10]],
    "question": [e["question"] for e in _val_list[:10]],
    "gold":     _gold[:10],
    "pred":     spider_preds[:10],
    "match":    [" ".join(g.lower().split()) == " ".join(p.lower().split())
                 for g, p in zip(_gold[:10], spider_preds[:10])],
})
display(_preview)

# Save predictions to /kaggle/working/ (downloadable)
pd.DataFrame({
    "question": [e["question"] for e in _val_list],
    "gold":     _gold,
    "pred":     spider_preds,
}).to_csv("/kaggle/working/spider_predictions.csv", index=False)
with open("/kaggle/working/spider_metrics.json", "w") as _f:
    json.dump(_results, _f, indent=2)
print("Saved: /kaggle/working/spider_predictions.csv")
print("Saved: /kaggle/working/spider_metrics.json")
print(f"Saved: {SPIDER_FINAL_DIR}/")

## 6. Interactive demo
Launches a Gradio app where you can type a question and a schema and get generated SQL, optionally executed against a small in-memory SQLite database.

In [ ]:
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"


def normalize_type(type_name):
    t = str(type_name).strip().lower()
    return "number" if t in {"number","real","integer","int","float","numeric"} else "text"


def _parse_multi_schema(schema_text):
    text  = str(schema_text or "").strip()
    db_id = "my_db"
    dm = re.search(r"db(?:atabase)?\s*:\s*(\w+)", text, re.I)
    if dm:
        db_id = dm.group(1)
    tables = {}
    for _line in text.split("\n"):
        _line = _line.strip()
        tm = (re.match(r"table\s*:\s*(\w+)\s*\(([^)]+)\)", _line, re.I) or
              re.match(r"(\w+)\s*\(([^)]+)\)", _line))
        if tm:
            tname = tm.group(1)
            cols  = []
            for _p in tm.group(2).split(","):
                _tok = _p.strip().split()
                if _tok:
                    cols.append((_tok[0], _tok[1] if len(_tok) > 1 else "text"))
            if cols:
                tables[tname] = cols
    return db_id, tables


def _schema_to_prompt(db_id, tables):
    parts = [
        f"{tn}(" + ", ".join(f"{c} {t}" for c, t in cols[:12]) + ")"
        for tn, cols in tables.items()
    ]
    return f"db: {db_id} | tables: " + " | ".join(parts)


def generate_sql(question, schema_text):
    try:
        q = str(question or "").strip()
        if not q:
            return "-- Please enter a question."
        db_id, tables = _parse_multi_schema(schema_text)
        if not tables:
            return "-- Could not parse schema.\nFormat: table: name (col type, ...)"
        prompt = (f"translate to SQL: question: {q} "
                  f"| {_schema_to_prompt(db_id, tables)}")
        enc = tokenizer(prompt, return_tensors="pt",
                        max_length=SPIDER_MAX_INPUT_LENGTH, truncation=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = spider_model.generate(
                **enc,
                max_length=SPIDER_MAX_TARGET_LENGTH,
                num_beams=4, early_stopping=True, no_repeat_ngram_size=3,
                decoder_start_token_id=spider_model.config.decoder_start_token_id,
                forced_eos_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0], skip_special_tokens=True).strip() or "-- Empty output."
    except Exception as exc:
        return f"-- ERROR: {type(exc).__name__}: {exc}"


def _execute_sql(sql, schema_text):
    db_id, tables = _parse_multi_schema(schema_text)
    BASE = [
        {"id":1,"name":"Alice","salary":120000.0,"dept_id":1,"age":34,
         "dept_name":"Sales","location":"Paris"},
        {"id":2,"name":"Bob",  "salary": 90000.0,"dept_id":2,"age":29,
         "dept_name":"Engineering","location":"Berlin"},
        {"id":3,"name":"Carol","salary":105000.0,"dept_id":1,"age":41,
         "dept_name":"Sales","location":"Madrid"},
    ]
    conn = sqlite3.connect(":memory:")
    try:
        for tn, cols in tables.items():
            col_defs = ", ".join(
                f'"{c}" {"REAL" if normalize_type(t) == "number" else "TEXT"}'
                for c, t in cols
            )
            conn.execute(f'CREATE TABLE "{tn}" ({col_defs})')
            ph = ",".join(["?"] * len(cols))
            for ri, base in enumerate(BASE, 1):
                row = []
                for cn, ct in cols:
                    v = base.get(cn.lower())
                    if v is None:
                        v = float(ri*10) if normalize_type(ct) == "number" else f"{cn}_{ri}"
                    row.append(v)
                conn.execute(f'INSERT INTO "{tn}" VALUES ({ph})', row)
        conn.commit()
        return pd.read_sql_query(sql, conn)
    finally:
        conn.close()


def gradio_predict(question, schema_text, execute_query):
    sql = generate_sql(question, schema_text)
    if sql.startswith("--") or not execute_query:
        return sql, pd.DataFrame({"info": ["Execution skipped."]})
    try:
        return sql, _execute_sql(sql, schema_text)
    except Exception as exc:
        return sql, pd.DataFrame({"error": [f"{type(exc).__name__}: {exc}"]})


_DEFAULT = (
    "table: employees (id INTEGER, name TEXT, salary REAL, dept_id INTEGER)\n"
    "table: departments (id INTEGER, dept_name TEXT, location TEXT)"
)
_EXAMPLES = [
    ["Who has the highest salary?",                   _DEFAULT, True],
    ["What is the name of the lowest paid employee?", _DEFAULT, True],
    ["Show average salary per department.",           _DEFAULT, True],
    ["Which departments have more than 1 employee?",  _DEFAULT, True],
    ["List employees ordered by salary descending.",  _DEFAULT, True],
    ["What is the total salary in Sales?",            _DEFAULT, True],
]

with gr.Blocks(title="WikiSQL + Spider Text-to-SQL") as demo:
    gr.Markdown("# WikiSQL + Spider -- Text-to-SQL Demo")
    gr.Markdown(
        "Trained on **WikiSQL** then **Spider**. "
        "Supports `ORDER BY`, `LIMIT`, `GROUP BY`, `HAVING`, and `JOIN`.\n\n"
        "Schema: one `table: name (col type, ...)` line per table."
    )
    with gr.Row():
        q_box  = gr.Textbox(label="Question", lines=2,
                             value="Who has the highest salary?")
        ex_box = gr.Checkbox(label="Execute on dummy DB", value=True)
    s_box   = gr.Textbox(label="Schema (multi-table)", lines=4, value=_DEFAULT)
    btn     = gr.Button("Generate SQL", variant="primary")
    sql_out = gr.Textbox(label="Generated SQL", lines=6)
    res_out = gr.Dataframe(label="Result (dummy data)")
    btn.click(fn=gradio_predict,
              inputs=[q_box, s_box, ex_box],
              outputs=[sql_out, res_out])
    gr.Examples(examples=_EXAMPLES, inputs=[q_box, s_box, ex_box],
                outputs=[sql_out, res_out],
                fn=gradio_predict, cache_examples=False)

demo.launch(share=True, debug=True)
